# DermArbiter — Case RAG Colab Ingest (Derm1M, 50K stratified subset)

**Owner:** Emre (Mahmut) — task M-2 
**Runtime:** Colab → T4 GPU (Runtime menu → Change runtime type → T4 GPU) 
**Wall-clock estimate:** ~50 minutes total (HF auth + manifest pull < 2 min, image download ~6–8 min, embed ~25–35 min on T4, retrieval test < 1 min, Drive upload ~3 min). 
**Output:** A populated ChromaDB collection at `data/chroma_cases/derm1m_cases` zipped to your Google Drive.

## Security — read this before pasting any tokens
We use **Colab Secrets** instead of hard-coding API keys: open the 🔑 panel on the left sidebar ("Secrets"), add two secrets, and toggle them on for this notebook:

| Secret name | Value | Notebook access |
|---|---|---|
| `HF_TOKEN` | Your HuggingFace token with Derm1M / MedGemma / DermoGPT access | ✅ |
| `GOOGLE_API_KEY` | Optional — only used downstream by run_e2e_gpu | ✅ |

If you've ever pasted a token into a chat or commit history, **rotate it immediately** before continuing — HF tokens: https://huggingface.co/settings/tokens — Google AI Studio keys: https://aistudio.google.com/app/apikey

## What this notebook does
1. Clone the DermArbiter repo into Colab and install Python deps.
2. Confirm T4 GPU + load secrets from Colab Secrets into `os.environ`.
3. Stratified pull of 50,000 Derm1M cases from HuggingFace (gated dataset). Only the chosen image files are downloaded — no full 80 GB snapshot.
4. Run `scripts/build_case_rag_index.py --source derm1m --max-cases 50000` — encodes all images with DermLIP-ViT-B-16 and upserts to ChromaDB.
5. Sanity-check by running 3 retrieval queries against the populated `CaseRAG` tool.
6. Zip `data/chroma_cases/` and copy it to Google Drive (so it survives Colab session end).

## Pre-requisites
- A GitHub remote for this repo. Push it as a private repo first (`gh repo create DermArbiter --private --source=. --push`) and paste the URL into the `REPO_URL` cell below.
- Derm1M access approved on HuggingFace: https://huggingface.co/datasets/redlessone/Derm1M (one-click usage agreement).

## 1. Clone repo + install dependencies

In [ ]:
REPO_URL = "https://github.com/<YOUR_GITHUB_USERNAME>/DermArbiter.git"  # ← EDIT THIS
REPO_BRANCH = "main"

import os, sys, subprocess

if not os.path.isdir("/content/DermArbiter"):
    !git clone -b $REPO_BRANCH $REPO_URL /content/DermArbiter
else:
    print("Repo already cloned, pulling latest...")
    !cd /content/DermArbiter && git pull

%cd /content/DermArbiter

!pip install -q -e .
!pip install -q huggingface_hub
!python scripts/setup_colab.py --check-only

## 2. Load secrets + confirm T4 GPU

Reads `HF_TOKEN` and `GOOGLE_API_KEY` from Colab Secrets and exports them into `os.environ` so the downstream scripts pick them up automatically. Never prints the tokens — only confirms they're set.

In [ ]:
from google.colab import userdata
import os

for key in ("HF_TOKEN", "GOOGLE_API_KEY"):
    try:
        os.environ[key] = userdata.get(key)
        print(f"✅ {key} loaded from Colab Secrets ({len(os.environ[key])} chars)")
    except userdata.SecretNotFoundError:
        if key == "HF_TOKEN":
            raise RuntimeError(
                "HF_TOKEN secret is required for Derm1M. "
                "Open the 🔑 panel (left sidebar) → Add new secret → name=HF_TOKEN → paste your token → toggle 'Notebook access'."
            )
        print(f"ℹ️  {key} not set (optional — used by downstream e2e scripts).")

# HuggingFace will also pick up HUGGINGFACE_HUB_TOKEN; mirror for safety.
os.environ.setdefault("HUGGINGFACE_HUB_TOKEN", os.environ["HF_TOKEN"])

!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
import torch
print("PyTorch CUDA available:", torch.cuda.is_available())
print("CUDA device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(none)")

## 3. Probe the Derm1M manifest (dry-run)

Pulls only the manifest CSV (~30 MB) and shows the disease-label distribution before downloading any images. Confirms your HF agreement works and gives you a chance to bail out cheaply if anything's off.

In [ ]:
!python scripts/build_case_rag_index.py \
    --source derm1m \
    --derm1m-split pretrain \
    --max-cases 50000 \
    --dry-run

import csv, collections
manifest = "data/derm1m/Derm1M_v2_pretrain.csv"
with open(manifest) as fh:
    rows = list(csv.DictReader(fh))
print(f"\nTotal manifest rows: {len(rows):,}")
labels = collections.Counter(r['disease_label'].strip() for r in rows if r.get('disease_label'))
print(f"Unique disease labels: {len(labels)}")
print("Top 15 labels:")
for label, n in labels.most_common(15):
    print(f"  {label:40s} {n:>6,d}")

## 4. Embed + upsert to ChromaDB (Derm1M 50K subset)

Stratified by `disease_label` so the subset preserves the rare-class distribution. Image download is scoped to the 50K filenames we'll actually use, not the full 80 GB. Encoding ~50K images at `--batch-size 64` on T4 takes ~25–35 min.

**The script is idempotent**: ChromaDB upsert is keyed by `case_id`, so if the runtime dies you can re-run this cell and it'll resume safely (the encoded vectors that already landed are preserved).

DermLIP weights (`hf-hub:redlessone/DermLIP_ViT-B-16`) are downloaded by `open_clip` on first call (~600 MB).

In [ ]:
import time
t0 = time.time()
!python scripts/build_case_rag_index.py \
    --source derm1m \
    --derm1m-split pretrain \
    --max-cases 50000 \
    --persist-dir data/chroma_cases \
    --collection derm1m_cases \
    --batch-size 64 \
    --device cuda
print(f"\nTotal ingest wall time: {(time.time() - t0)/60:.1f} min")

!du -sh data/chroma_cases/
!du -sh data/derm1m/

## 5. Retrieval smoke test

Query the populated `CaseRAG` tool with 3 random Derm1M images. We expect the top-1 retrieved case to share the same `disease_label` as the query in most cases.

In [ ]:
from pathlib import Path
import csv, random
from dermarbiter.tools.case_rag import CaseRAG

rows = list(csv.DictReader(open("data/derm1m/Derm1M_v2_pretrain.csv")))
rng = random.Random(0)
# Filter to rows whose image was actually downloaded (in our 50K subset)
available = [r for r in rows if (Path("data/derm1m") / r['filename']).exists()]
print(f"Images on disk: {len(available):,}")
queries = rng.sample(available, 3)

tool = CaseRAG(
    chroma_persist_dir="data/chroma_cases",
    collection_name="derm1m_cases",
)

for q in queries:
    img = f"data/derm1m/{q['filename']}"
    out = tool.run(image_path=img, query="top-5 similar cases")
    print(f"\n── Query: {q['filename']}  gt = {q['disease_label']}")
    print(f"   raw_text: {out.raw_text[:160]}")
    for m in out.result.get("similar_cases", [])[:5]:
        print(f"     - {m.get('case_id', '?'):20s}  dx={m.get('diagnosis', '?'):28s}  dist={m.get('distance', 0):.3f}")

## 6. Persist ChromaDB to Google Drive

Colab sessions are ephemeral — without this step you'll lose the index when the runtime disconnects. Zips `data/chroma_cases/` and drops it in your Drive. 
Download that file and unzip into the repo at `data/chroma_cases/` to use locally.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!mkdir -p /content/drive/MyDrive/dermarbiter
!cd data && zip -qr /content/drive/MyDrive/dermarbiter/chroma_cases_derm1m_50k.zip chroma_cases
!ls -lh /content/drive/MyDrive/dermarbiter/chroma_cases_derm1m_50k.zip

## Next steps

1. **Download** `chroma_cases_derm1m_50k.zip` from your Drive and `unzip` it into the local repo at `data/chroma_cases/`.
2. **Smoke test locally**:
   ```python
   from dermarbiter.tools.case_rag import CaseRAG
   tool = CaseRAG(chroma_persist_dir="data/chroma_cases", collection_name="derm1m_cases")
   print(tool.run(image_path="<some-derm-image>.jpg").raw_text)
   ```
3. **Mark M-2 complete** and proceed to O-1 (real model integration) — `scripts/run_e2e_gpu.py` will use the populated CaseRAG plus the real Gemini/MedGemma/DermoGPT models.
4. **(Optional later)** Re-run this notebook with `--max-cases 100000` or no `--max-cases` to grow the index. ChromaDB upserts are idempotent on `case_id`, so the existing 50K vectors are preserved.